# Phase 1: Federated Discovery, Deduplication & PRISMA Screening
**Nexus Scholar Interactive Research Suite**

This notebook demonstrates how to execute federated literature discovery, perform 2-tier PID deduplication, verify documents, and apply PRISMA 2020 heuristic screening.

In [ ]:
import asyncio
import json
from pathlib import Path
import pandas as pd
from scholar_protocol.models import ResearchProtocol
from scholar_search.engine import SearchEngine
from scholar_search.protocol_adapter import compile_protocol_search
from scholar_search.dedup import Deduplicator
from scholar_search.verifier import DocumentVerifier
from scholar_search.screening import evaluate_heuristic_screening, partition_screening_results
from scholar_search.export import Exporter

## 1. Compile Search Strategy from Protocol

In [ ]:
proto_path = Path("protocol.json")
if not proto_path.exists():
    print("Run 00_research_inception.ipynb first to generate protocol.json")
else:
    query, providers = compile_protocol_search(proto_path)
    query.max_results = 10
    print(f"Search Query: {query.text}")
    print(f"Target Providers: {[p.name for p in providers]}")

## 2. Execute Federated Search & 2-Tier Deduplication

In [ ]:
async def run_discovery():
    engine = SearchEngine(providers=providers)
    docs = await engine.search_all(query, dedup=False)
    await engine.close()
    return docs

discovered_docs = asyncio.run(run_discovery())
print(f"Discovered {len(discovered_docs)} candidate documents across providers.")

deduplicator = Deduplicator()
clusters = deduplicator.deduplicate(discovered_docs)
unique_docs = [c.representative for c in clusters]
print(f"Deduplicated into {len(unique_docs)} unique document clusters.")

## 3. Systematic PRISMA 2020 Screening

In [ ]:
protocol_data = json.loads(proto_path.read_text(encoding="utf-8"))
decisions = [evaluate_heuristic_screening(d, protocol_data) for d in unique_docs]
included, excluded, conflicts, report = partition_screening_results(unique_docs, decisions)

print(f"PRISMA Screening Results: {len(included)} Included, {len(excluded)} Excluded, {len(conflicts)} Conflicts")

# Display Interactive Table of Included Papers
df_inc = pd.DataFrame([
    {
        "Workspace ID": p.get("workspace_id"),
        "Title": p.get("title"),
        "Year": p.get("year"),
        "DOI": p.get("external_ids", {}).get("doi"),
        "Reasoning": p.get("screening", {}).get("screening_reasoning")
    }
    for p in included
])
df_inc.head()